In [ ]:
# # If you are working in Google Colab
# # Uncomment this cell and run it in order to clone the repository.


# import os
# if not os.path.exists("qest-hack26"):
#     !git clone https://github.com/jeanfredericlaprade/qest-hack26.git

# # Move into the repo folder so `import utils` works
# %cd qest-hack26

# # Install dependencies
# !pip install -q -r requirements.txt

# Problem 1: Coastal surveillance network maintenance

# Importing the required libraries

The following cell holds the necessary imports to execute this notebook.<br>
You are welcomed to modify this notebook as you see fit.

In [ ]:
# Displaying the problem graph
import matplotlib.pyplot as plt
import networkx as nx

# Matrix computation and classical optimization
import numpy as np
from scipy.optimize import minimize

# Quantum circuit creation
from qiskit import QuantumCircuit

# Structure needed for writing Hamiltonians
from qiskit.quantum_info import SparsePauliOp

# Creation method to build QAOA's circuit which will need to be optimize
from qiskit.circuit.library import QAOAAnsatz

# Quantum circuit execution tools
from qiskit_ibm_runtime.executor_estimator import Estimator
from qiskit_ibm_runtime.executor_sampler import Sampler

# Transpilation of quantum circuits for IBM's quantum computers and simulators
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager

# Quantum computer simulator
from qiskit_aer import AerSimulator

# Visualization of probability distributions
from qiskit.visualization import plot_histogram

# Introduction
A coastal surveillance network must schedule the maintenance of its radar and sensor stations, requiring the stations to be powered down during one of two period.<br>
This simplified scenario lets you experiment with solving a combinatorial optimization problem using the **QAOA** quantum algorithm.

## Problem statement
> The network has five radar or sensor stations whose maintenance must be scheduled to happen during one of two periods. If two stations have overlapping coverage, they should be assigned to different periods whenever possible in order to minimize surveillance blind spots.

Consider the following stations :
- Coastal radar station A (S0)
- Harbor sensor station B (S1)
- Northern observation station C (S2)
- Island radar station D (S3)
- Coastal sensor station E (S4)

To model this problem, we represent each station as a node in a graph. An edge between two nodes indicates the corresponding stations have overlapping coverage.

<p align="center">
<img src="" alt="FIXME" width="250"/>
</p>

## Displaying the graph with Python
The [Networkx library](https://networkx.org/en/) is used to visualize the graph corresponding to this problem. The option to add and display weights on edges is available, but it is disabled by default in the code.

Viewing the problem graph is optional, but may potentially help you in solving future challenges.

In [ ]:
# Instanciate a graph using Networkx
G = nx.Graph()

# Add the edges of the graph and their weights, if needed, with format [(node_A, node_B, edge_weight_AB), ...]
G.add_weighted_edges_from([(0, 1, 1), (1, 2, 1), (2, 3, 1), (3, 0, 1), (0, 4, 1), (4, 1, 1)])

# Define the positions of the nodes for drawing the graph
pos = {1: (-1, 0), 0: (1, 0), 3: (1, -1), 2: (-1, -1), 4: (0, 1)}

# Define visualization options
options = {"node_size": 1000, "node_color": "white", "edgecolors": "black", "linewidths": 2, "width": 2}

# Draw the graph
nx.draw_networkx(G, pos, **options)
# if needed, display the edge weights
# nx.draw_networkx_edge_labels(G, pos, nx.get_edge_attributes(G, "weight"))

# Display the graph visualization
ax = plt.gca()
ax.margins(0.20)
plt.axis("off")
plt.show()

## Objective
Your mission is to optimize the hyperparameters of the QAOA circuit in order to solve this problem efficiently. The solution pipeline with QAOA is already provided, but you are asked to find the best circuit parameters, as well as the optimal number of circuit layers.

### Expected deliverables:
- A set of optimal parameters for the QAOA circuit.
- The number of layers in the QAOA circuit.

#### Deliverable submission format:
A file in `.npz` format is expected. You can generate this file with the required information using the `save_res` method available in `utils.py`.

### Grading
Part of the grading will be done by running your QAOA circuit with the number of layers and parameters provided. A score will be calculated from the solution obtained using the predefined scoring method `compute_score` available in `utils.py`.

The judges will also evaluate the quality of the code and the originality of your parameter optimization for QAOA.

# The cost function

Before solving the problem using QAOA, we must first define a cost function that represents our optimization objective.

## Qubits and expectation value of observables

The coastal surveillance network maintenance problem must be reformulated in terms of qubits and quantum logic gates in order to be solved by a quantum algorithm such as **QAOA**. Each node in the graph represents a surveillance station, and we use one qubit to encode each node. Thus, the number of qubits in our circuit corresponds to the number of nodes (or stations) in the graph.

To capture the relationships between nodes, we define the interactions between qubits based on the edges of the graph. For example, a Pauli operator such as $IIIZZ$ encodes an edge between qubits (nodes) 0 and 1, where each Pauli matrix $Z_i$ corresponds to a measurement on qubit $i$. The two possible qubit values represent the two maintenance periods.

When stations represented by qubits 0 and 1 are assigned to the same period, the observable $IIIZZ$ associated with this edge will take the value $+1$. If the stations are assigned to different periods, the observable will take the value $-1$. This formulation allows us to evaluate whether two nodes (stations) are correctly separated.

If necessary, refer to the presentation for reminders on measuring observables.

## Expressing the Hamiltonian

The Hamiltonian, which represents the cost function of the problem, is expressed as the sum of the Pauli operators associated with each pair of qubits connected by an edge in the graph. For two connected qubits $i$ and $j$, the interaction is modeled by the term $Z_i Z_j$.

Thus, the Hamiltonian for the entire graph is:

$$
H = \sum_{(i,j) \in E} Z_i Z_j
$$

where $E$ is the set of edges in the graph. This sum covers all pairs of nodes $i$ and $j$ that share an edge, i.e. all station pairs with overlapping coverage.

> The objective is to minimize this function in order to assign stations to two periods so that overlapping coverage is separated where possible.

**Note :** This corresponds to a Max-Cut problem.

We use the `SparsePauliOp` structure to instantiate the Hamiltonian.

In [ ]:
# Instanciate the SparsePauliOp as a sum of Pauli chains multiplied by a certain coefficient
H = SparsePauliOp.from_list(
    [("IIIZZ", 1.0), ("IIZZI", 1.0), ("IZZII", 1.0), ("IZIIZ", 1.0), ("ZIIIZ", 1.0), ("ZIIZI", 1.0)]
)
# Display the SparsePauliOp
print(H)

# Classical verification

The problem can be represented in matrix form. Given its small size, it is possible to diagonalize this matrix to obtain an exact solution. You can use this diagonalization method to verify the validity of your quantum solution.

**Note** : This method can be found in `utils.py` and can be reused to check your solutions to the following problems as well.

In [ ]:
def compute_exact_sols(hamiltonian: SparsePauliOp) -> tuple[float, list[str]]:
    """
    Computes the solutions to the input Hamiltonian classically.
    This is done by diagonalizing the Hamiltonian matrix.

    Args:
        hamiltonian (SparsePauliOp): Hamiltonian to diagonalize, expressed as a sum of Pauli operators.

    Returns:
        tuple[float, list[str]]:
            - Minimal cost obtained (float)
            - List of binary solutions associated with the minimal cost.
    """
    # Transforms the hamiltonian into a matrix
    hamiltonian_mat = np.array(hamiltonian.to_matrix())

    # Diagonalize the matrix to extract the eigenvalues and eigenvectors
    eigenvals, eigenvects = np.linalg.eig(hamiltonian_mat)

    # Smallest eigenvalues indices
    min_eigenval = np.where(eigenvals == np.min(eigenvals))[0]

    # Solutions associated to the smallest eigenvalues
    binary_sols = [bin(idx).lstrip("-0b").zfill(hamiltonian.num_qubits) for idx in min_eigenval]

    # Cost and binary representation of the best solutions
    return eigenvals[min_eigenval][0].real, binary_sols

In [ ]:
# Verify that the Hamiltonian gives the expected solution
minimum_cost, binary_sols = compute_exact_sols(hamiltonian=H)

# Display the minimal cost
print("Minimum cost : ", minimum_cost)

# Display the binary solutions (to read from right to left, from qubit 0 to qubit 4)
for i, sol in enumerate(binary_sols):
    print("Minimum cost solution {idx} : {solution}".format(idx=i, solution=sol))

# Solving the problem with QAOA

QAOA (Quantum Approximate Optimization Algorithm) is a variationnal quantum algorithm, used here to solve our combinatorial optimization problem. For theoretical details of the algorithm, please refer to the presentation.

In this first part of the challenge, you must adjust the parameters of your hybrid execution to effectively solve the coastal surveillance network maintenance problem. The parameters to be adjusted are:

- The number of layers in your QAOA circuit.
- The rotation angles of the QAOA circuit.
- The options of the classical optimizer used to adjust the angles of the QAOA circuit.

Your goal is to find the best combination of layers and angles to consistently obtain solutions to the MaxCut problem for multiple runs of the QAOA circuit.

## QAOA circuit

In the following cells, you will find a proposed structure for building the QAOA circuit based on the Hamiltonian of the problem. Experiment with the number of layers to improve the approximation of the solution.

**Note** : Too few layers may not provide a stable solution, while too many layers significantly increases execution time. The ideal solution is to find a balance that effectively amplifies the solutions without unnecessarily complicating the simulation, in order to remain within reasonable time limits for execution on a laptop.

In [ ]:
# Identify the number of layers (repetitions) in QAOA
num_layers = 1  # Modify as you see fit

# Instanciate a QAOA circuit with Qiskit
circuit_qaoa = QAOAAnsatz(H, reps=num_layers)

# If desired, display QAOA circuit
circuit_qaoa.decompose(reps=1).draw(output="mpl", scale=0.6)

In [ ]:
# Define a simulator to compute the expectation values of our observables
backend = AerSimulator()  # You can change to another simulator if you want, or to a real quantum processor if available
estimator = Estimator(mode=backend)

# Define the transpilation tool expected by Qiskit
pm = generate_preset_pass_manager(backend=backend, optimization_level=1)

In [ ]:
# Define the classical cost function to optimize
def cost_fonction(
    params: list[complex], estimator: Estimator, circuit: QuantumCircuit, hamiltonian: SparsePauliOp
) -> float:
    """
    Cost function that calculates the expectation value of an observable (Hamiltonian) for a given state (circuit).
    The circuit is parameterized and the parameters are defined in the params vector.
    Everything is evaluated using the estimator.

    Args:
        params (list[complex]): list of parameters to assign to the specified circuit.
        estimator (Estimator): Simulator used to estimate the expectation value.
        circuit (QuantumCircuit): Parametrized circuit of QAOA.
        hamiltonian (SparsePauliOp): Observable describing the cost function of the problem.

    Returns:
        float: Cost for the parameters provided as input
    """
    isa_psi = pm.run(circuit)
    isa_observables = hamiltonian.apply_layout(isa_psi.layout)

    job = estimator.run([(isa_psi, isa_observables, params)])
    cost = job.result()[0].data.evs

    return cost

## QAOA circuit parameters

The following cell allows you to perform classic optimization of the QAOA circuit angles. These parameters (angles) are initialized with a vector of zeros, but it may be *very* useful to explore other initializations to improve the optimization results.

### Classical optimizer

The **Cobyla** optimizer is proposed to adjust the parameters using the previously defined cost function. You are free to modify Cobyla's options or use another optimizer if you find it more appropriate.

Some optimizers without gradient computation are implemented directly in `Qiskit`.

In [ ]:
# Initializing QAOA parameters to zeros
params_init = np.zeros(circuit_qaoa.num_parameters)

# Classical optimization of QAOA parameters with the help of Scipy
opt_res = minimize(cost_fonction, params_init, args=(estimator, circuit_qaoa, H), method="COBYLA")

# Extracting information following the optimization
opt_cost = opt_res.fun  # Optimal cost found
opt_params = opt_res.x  # Optimal parameters found

# Displaying the generated results
print("Optimal cost found :", opt_cost)
print("Optimal parameters found :", opt_params)

# Evaluation of the solution

After determining a number of layers and a parameter vector, a **sampler** is used to generate the probability distribution associated with the QAOA circuit. The goal is to verify that the correct solutions to our problem are significantly amplified in this distribution.

This is one of the ways in which your score will be determined.

In [ ]:
# Instanciate a Sampler to obtain a probability distribution
sampler = Sampler(mode=backend)

# Compute the probability distribution with the optimal parameters you found during optimization.
circuit_qaoa_copy = circuit_qaoa.decompose(reps=2).copy()
circuit_qaoa_copy.measure_all()
circuit_qaoa_copy.draw()
counts = sampler.run([(circuit_qaoa_copy, opt_params)]).result()[0].data.meas.get_counts()

# Display the probability distribution
plot_histogram(counts)

# Save your results

Finally, don't forget to save your results using the `save_res` method available in `utils.py`.

Please name your file `<your team>_problem_1.npz`. You can check the content of your `.npz` file using the `read_res` method also available in `utils.py`.